# 📄 Notebook 1 — Document Processing for RAG

**DocuMind AI Portfolio Project**

This notebook covers:
1. Loading PDF, DOCX, and TXT documents
2. Exploring document metadata
3. Text cleaning & preprocessing
4. Chunking strategies comparison
5. Chunk size impact analysis
6. Visualising chunk distribution

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0d1117'
matplotlib.rcParams['axes.facecolor'] = '#161b22'
matplotlib.rcParams['text.color'] = '#e6edf3'
matplotlib.rcParams['axes.labelcolor'] = '#e6edf3'
matplotlib.rcParams['xtick.color'] = '#e6edf3'
matplotlib.rcParams['ytick.color'] = '#e6edf3'
import numpy as np
import pandas as pd

print('✅ Imports successful')

## 1. Load Configuration

In [ ]:
from src.document_processor import DocumentProcessor, load_config

config = load_config('config/config.yaml')
processor = DocumentProcessor(config)

print(f"Chunk size: {processor.chunk_size}")
print(f"Chunk overlap: {processor.chunk_overlap}")
print(f"Separators: {processor.separators[:4]}...")

## 2. Load Documents from Multiple File Types

In [ ]:
# Load all sample documents
docs = processor.load_multiple_documents('data/raw/sample_docs')
txt_docs = processor.load_document('data/raw/sample_texts/knowledge_base.txt')
all_docs = docs + txt_docs

print(f'Total document pages/sections loaded: {len(all_docs)}')

# Inspect first document
if all_docs:
    d = all_docs[0]
    print(f"\nFirst document:\n  Source: {d.metadata.get('filename')}")
    print(f"  File type: {d.metadata.get('file_type')}")
    print(f"  Page: {d.metadata.get('page', 'N/A')}")
    print(f"  Content preview: {d.page_content[:150]}...")

## 3. Explore Document Metadata

In [ ]:
# Build a metadata DataFrame
rows = []
for doc in all_docs:
    rows.append({
        'filename': doc.metadata.get('filename', 'unknown'),
        'file_type': doc.metadata.get('file_type', 'unknown'),
        'page': doc.metadata.get('page', 0),
        'word_count': len(doc.page_content.split()),
        'char_count': len(doc.page_content),
    })

df = pd.DataFrame(rows)
print(df.groupby('filename')[['word_count','char_count']].agg(['sum','mean']).round(1))

## 4. Text Cleaning Demo

In [ ]:
dirty_text = """
  This   is  a   DIRTY   text\x00  example.
  
  
  It has   extra whitespace,  unicode issues: \u00e9\u00e0,
  and    page numbers like:
  
  42
  
  Should be cleaned up nicely.
"""

cleaned = processor.clean_text(dirty_text)
print('BEFORE:')
print(repr(dirty_text[:100]))
print('\nAFTER:')
print(repr(cleaned))

## 5. Chunking Strategies Comparison

In [ ]:
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
)

sample_text = "\n\n".join(d.page_content for d in all_docs[:3])

# Strategy 1: Fixed size
fixed_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0, separator=' ')
fixed_chunks = fixed_splitter.split_text(sample_text)

# Strategy 2: Recursive (DocuMind default)
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200,
    separators=['\n\n', '\n', '.', '!', '?', ',', ' ']
)
recursive_chunks = recursive_splitter.split_text(sample_text)

# Strategy 3: Large chunks (less overlap)
large_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=100)
large_chunks = large_splitter.split_text(sample_text)

results = {
    'Fixed-Size (1000, 0 overlap)': fixed_chunks,
    'Recursive (1000, 200 overlap)': recursive_chunks,
    'Recursive (2000, 100 overlap)': large_chunks,
}

for name, chunks in results.items():
    sizes = [len(c) for c in chunks]
    print(f"{name:45s} → {len(chunks):3d} chunks | avg={np.mean(sizes):5.0f} chars | min={min(sizes):4d} | max={max(sizes):4d}")

## 6. Chunk Size Impact Analysis

In [ ]:
chunk_sizes = [256, 512, 750, 1000, 1500, 2000]
chunk_counts = []
avg_sizes = []

for cs in chunk_sizes:
    sp = RecursiveCharacterTextSplitter(chunk_size=cs, chunk_overlap=int(cs * 0.2))
    chunks = sp.split_text(sample_text)
    chunk_counts.append(len(chunks))
    avg_sizes.append(np.mean([len(c) for c in chunks]))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(chunk_sizes, chunk_counts, 'o-', color='#7C3AED', linewidth=2, markersize=7)
axes[0].set_title('Chunk Size vs Number of Chunks', color='#e6edf3')
axes[0].set_xlabel('Chunk Size (chars)'); axes[0].set_ylabel('Number of Chunks')
axes[0].grid(True, alpha=0.3)

axes[1].bar(chunk_sizes, avg_sizes, color='#06B6D4', alpha=0.8, width=150)
axes[1].set_title('Chunk Size vs Avg Actual Size', color='#e6edf3')
axes[1].set_xlabel('Target Chunk Size'); axes[1].set_ylabel('Avg Actual Size (chars)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()
print('\n💡 Insight: Larger chunks = fewer retrievals but may include irrelevant context.')

## 7. Final Chunking & Stats

In [ ]:
# Chunk all documents with DocuMind default settings
chunks = processor.chunk_documents(all_docs)
chunks = processor.deduplicate_documents(chunks)
stats = processor.get_document_stats(chunks)

print('📊 Document Statistics after chunking:')
for k, v in stats.items():
    print(f'  {k}: {v}')

# Chunk size distribution
sizes = [len(c.page_content) for c in chunks]
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(sizes, bins=30, color='#7C3AED', alpha=0.8, edgecolor='#30363d')
ax.axvline(np.mean(sizes), color='#06B6D4', linestyle='--', label=f'Mean: {np.mean(sizes):.0f}')
ax.set_title('Chunk Size Distribution', color='#e6edf3')
ax.set_xlabel('Chunk Size (chars)'); ax.set_ylabel('Count')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()